In [1]:
from pathlib import Path
from collections import Counter
import csv
import sys

import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    ANSWERS_DIR,
    INSIDERS_CSV,
    FILE_CSV,
    RESULTS_DIR,
)


# ============================================================
# LOAD CERT R4.2 GROUND TRUTH
# ============================================================

insiders = pd.read_csv(
    INSIDERS_CSV,
    dtype=str,
)

insiders["dataset"] = pd.to_numeric(
    insiders["dataset"],
    errors="coerce",
)

insiders["scenario"] = pd.to_numeric(
    insiders["scenario"],
    errors="coerce",
).astype("Int64")

r42 = insiders[
    insiders["dataset"] == 4.2
].copy()


# ============================================================
# AUDIT INCIDENT EVENT TYPES
# ============================================================

results = []

print("=" * 70)
print("CERT R4.2 INCIDENT EVENT AUDIT")
print("=" * 70)

for _, row in r42.iterrows():

    scenario = int(row["scenario"])
    user = row["user"]
    filename = row["details"]

    incident_file = (
        ANSWERS_DIR
        / f"r4.2-{scenario}"
        / filename
    )

    if not incident_file.exists():
        raise FileNotFoundError(
            f"Incident file not found:\n{incident_file}"
        )

    counts = Counter()

    first_time = None
    last_time = None

    with incident_file.open(
        "r",
        encoding="utf-8",
        errors="replace",
        newline="",
    ) as file:

        reader = csv.reader(file)

        for parts in reader:

            if not parts:
                continue

            event_type = (
                parts[0]
                .strip()
                .lower()
            )

            counts[event_type] += 1

            # Expected format:
            # event_type, id, date, user, pc, ...
            if len(parts) >= 3:

                timestamp = pd.to_datetime(
                    parts[2],
                    format="%m/%d/%Y %H:%M:%S",
                    errors="coerce",
                )

                if pd.notna(timestamp):

                    if (
                        first_time is None
                        or timestamp < first_time
                    ):
                        first_time = timestamp

                    if (
                        last_time is None
                        or timestamp > last_time
                    ):
                        last_time = timestamp

    results.append(
        {
            "scenario": scenario,
            "user": user,
            "logon": counts.get("logon", 0),
            "device": counts.get("device", 0),
            "file": counts.get("file", 0),
            "http": counts.get("http", 0),
            "email": counts.get("email", 0),
            "total_events": sum(counts.values()),
            "first_event": first_time,
            "last_event": last_time,
        }
    )


df = pd.DataFrame(results)

event_columns = [
    "logon",
    "device",
    "file",
    "http",
    "email",
    "total_events",
]


# ============================================================
# RESULTS
# ============================================================

print(f"\nIncidents audited : {len(df)}")
print(
    f"Unique users      : "
    f"{df['user'].nunique()}"
)


print("\nEvent totals by scenario:")

event_totals = (
    df
    .groupby("scenario")[event_columns]
    .sum()
)

display(event_totals)


print("\nAverage events per incident:")

event_averages = (
    df
    .groupby("scenario")[event_columns]
    .mean()
    .round(2)
)

display(event_averages)


print("\nUsers with each event type:")

coverage_rows = []

for scenario in sorted(
    df["scenario"].unique()
):

    temp = df[
        df["scenario"] == scenario
    ]

    for event_type in [
        "logon",
        "device",
        "file",
        "http",
        "email",
    ]:

        users_with_event = int(
            (temp[event_type] > 0).sum()
        )

        coverage_rows.append(
            {
                "scenario": scenario,
                "event_type": event_type,
                "users_with_event": users_with_event,
                "total_users": len(temp),
                "coverage_percent": round(
                    100
                    * users_with_event
                    / len(temp),
                    2,
                ),
            }
        )


coverage = pd.DataFrame(
    coverage_rows
)

display(coverage)


print("\nFirst 10 incident summaries:")

display(
    df.head(10)
)


# ============================================================
# SAVE AUDIT RESULT
# ============================================================

output_file = (
    RESULTS_DIR
    / "incident_event_audit.csv"
)

df.to_csv(
    output_file,
    index=False,
)

print(
    f"\n✅ INCIDENT EVENT AUDIT COMPLETE"
)

print(
    f"Results saved to:\n{output_file}"
)

CERT R4.2 INCIDENT EVENT AUDIT

Incidents audited : 70
Unique users      : 70

Event totals by scenario:


,logon,device,file,http,email,total_events
scenario,,,,,,
1,138,138,0,69,0,345
2,0,2607,0,3738,420,6765
3,60,40,10,53,50,213



Average events per incident:


,logon,device,file,http,email,total_events
scenario,,,,,,
1,4.6,4.6,0.0,2.3,0.0,11.5
2,0.0,86.9,0.0,124.6,14.0,225.5
3,6.0,4.0,1.0,5.3,5.0,21.3



Users with each event type:


,scenario,event_type,users_with_event,total_users,coverage_percent
0,1,logon,30,30,100.0
1,1,device,30,30,100.0
2,1,file,0,30,0.0
3,1,http,30,30,100.0
4,1,email,0,30,0.0
5,2,logon,0,30,0.0
6,2,device,30,30,100.0
7,2,file,0,30,0.0
8,2,http,30,30,100.0
9,2,email,30,30,100.0



First 10 incident summaries:


,scenario,user,logon,device,file,http,email,total_events,first_event,last_event
0,1,AAM0658,4,4,0,2,0,10,2010-10-23 01:34:19,2010-10-29 05:23:28
1,1,AJR0932,4,4,0,2,0,10,2010-09-10 19:12:01,2010-09-18 02:02:51
2,1,BDV0168,6,6,0,3,0,15,2010-07-30 19:56:44,2010-08-10 05:16:41
3,1,BIH0745,2,2,0,1,0,5,2010-07-13 20:15:23,2010-07-13 21:20:44
4,1,BLS0678,4,4,0,2,0,10,2010-09-21 01:16:22,2010-09-30 04:48:19
5,1,BTL0226,6,6,0,3,0,15,2010-10-06 22:25:52,2010-10-14 06:43:29
6,1,CAH0936,4,4,0,2,0,10,2010-08-11 04:00:08,2010-08-12 23:56:19
7,1,DCH0843,2,2,0,1,0,5,2011-02-04 07:08:00,2011-02-04 07:36:05
8,1,EHB0824,8,8,0,4,0,20,2010-07-22 21:48:43,2010-07-29 01:08:41
9,1,EHD0584,6,6,0,3,0,15,2010-10-02 03:46:16,2010-10-08 22:26:26



✅ INCIDENT EVENT AUDIT COMPLETE
Results saved to:
C:\Users\HP\Documents\FYP\Data_engineering\results\incident_event_audit.csv


In [3]:
# ============================================================
# CASE STUDY: AAF0535
# NON-HTTP MALICIOUS EVENTS
# ============================================================

user = "AAF0535"
scenario = 2

incident_file = (
    ANSWERS_DIR
    / f"r4.2-{scenario}"
    / f"r4.2-{scenario}-{user}.csv"
)

print("=" * 70)
print(f"NON-HTTP INCIDENT EVENTS: {user}")
print("=" * 70)

if not incident_file.exists():
    raise FileNotFoundError(
        f"Incident file not found:\n{incident_file}"
    )


non_http_events = []

with incident_file.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline="",
) as file:

    reader = csv.reader(file)

    for row in reader:

        if not row:
            continue

        event_type = (
            row[0]
            .strip()
            .lower()
        )

        if event_type != "http":
            non_http_events.append(row)


print(
    f"\nNon-HTTP events found: "
    f"{len(non_http_events)}\n"
)

for row in non_http_events:
    print(",".join(row))


print(
    "\n✅ NON-HTTP EVENT INSPECTION COMPLETE"
)

NON-HTTP INCIDENT EVENTS: AAF0535

Non-HTTP events found: 99

email,{O9H3-T6RD70KY-9321LRAL},07/06/2010 11:27:45,AAF0535,PC-2408,Jaime_Carey@raytheon.com,Athena.Amelia.Foreman@dtaa.com,,Athena.Amelia.Foreman@dtaa.com,25030,0,hours years growth platform self contribute years skills multiple process degree people interface required years platform resume expert expert responsibilities growth initiative dynamic job benefits responsibilities initiative compensation salary expert opening on-line expert technologies compensation guidance part-time degree platform required multiple degree hours process multitask multiple report start process growth skills benefits visual dynamic strong years required sales industry resume technologies skills multitask years passion start job industry visual
email,{W5S2-M0UX21CW-4401FULS},07/09/2010 09:52:54,AAF0535,PC-2408,Jaime_Carey@raytheon.com,,Athena.Amelia.Foreman@dtaa.com,Athena.Amelia.Foreman@dtaa.com,40776,0,expert years management benefits required r

In [5]:
import duckdb


# ============================================================
# CASE STUDY: AAF0535 FILE ACTIVITY
# ============================================================

file_path = FILE_CSV.resolve().as_posix()

con = duckdb.connect()

result = con.execute(f"""
    SELECT
        try_strptime(
            date,
            '%m/%d/%Y %H:%M:%S'
        ) AS timestamp,

        user,
        pc,
        filename

    FROM read_csv(
        '{file_path}',
        header = true,
        all_varchar = true
    )

    WHERE user = 'AAF0535'

      AND try_strptime(
            date,
            '%m/%d/%Y %H:%M:%S'
          )
          BETWEEN
              TIMESTAMP '2010-08-01 00:00:00'
              AND
              TIMESTAMP '2010-08-21 23:59:59'

    ORDER BY timestamp
""").df()

con.close()


print("=" * 70)
print("AAF0535 FILE ACTIVITY")
print("=" * 70)

print(
    "\nObservation window : "
    "2010-08-01 → 2010-08-21"
)

print(
    f"Number of file events : "
    f"{len(result)}"
)

print("\nFile activity:")

display(result)

print(
    "\n✅ FILE ACTIVITY INSPECTION COMPLETE"
)

AAF0535 FILE ACTIVITY

Observation window : 2010-08-01 → 2010-08-21
Number of file events : 58

File activity:


,timestamp,user,pc,filename
0,2010-08-02 13:21:00,AAF0535,PC-2408,GS1ZDYRN.doc
1,2010-08-02 16:05:55,AAF0535,PC-2408,M9SLI981.doc
2,2010-08-02 16:30:27,AAF0535,PC-2408,0RIS0KGE.doc
3,2010-08-02 16:33:19,AAF0535,PC-2408,MUZJ1OJU.doc
4,2010-08-02 16:37:43,AAF0535,PC-2408,IQ8FARGK.doc
5,2010-08-03 13:56:03,AAF0535,PC-2408,G76KA12K.jpg
6,2010-08-03 14:04:19,AAF0535,PC-2408,GJZPGNUG.doc
7,2010-08-03 14:16:51,AAF0535,PC-2408,XDZH0X8U.doc
8,2010-08-03 14:16:53,AAF0535,PC-2408,P87QYUMB.doc
9,2010-08-03 14:23:00,AAF0535,PC-2408,BCM0Y1JF.zip



✅ FILE ACTIVITY INSPECTION COMPLETE


In [7]:
# ============================================================
# AAF0535 DAILY FILE ACTIVITY
# ============================================================

file_path = FILE_CSV.resolve().as_posix()

con = duckdb.connect()

daily = con.execute(f"""
    SELECT
        CAST(
            try_strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        ) AS day,

        COUNT(*) AS files_copied

    FROM read_csv(
        '{file_path}',
        header = true,
        all_varchar = true
    )

    WHERE user = 'AAF0535'

    GROUP BY day
    ORDER BY day
""").df()

con.close()


print("=" * 70)
print("AAF0535 DAILY FILE ACTIVITY")
print("=" * 70)

print(
    f"\nActive file days : "
    f"{len(daily)}"
)

print(
    f"Total file events: "
    f"{daily['files_copied'].sum():,}"
)

print("\nDaily activity:")

display(daily)

print(
    "\n✅ DAILY FILE ACTIVITY COMPLETE"
)

AAF0535 DAILY FILE ACTIVITY

Active file days : 112
Total file events: 357

Daily activity:


,day,files_copied
0,2010-01-05,1
1,2010-01-06,5
2,2010-01-07,1
3,2010-01-08,3
4,2010-01-12,2
...,...,...
107,2010-08-16,1
108,2010-08-17,5
109,2010-08-20,3
110,2010-08-23,5



✅ DAILY FILE ACTIVITY COMPLETE
